# From MLE to MAP: When Priors Matter

Back to the popcorn on the floor — with a new suspect: a **popcorn-throwing contest**. It generates popcorn on the floor *almost for sure*, beating movies at pure likelihood. Yet something says movies should still win: a popcorn-throwing contest is a wildly unlikely event *on its own*. The fix is to multiply the likelihood by the scenario's own probability — the **prior** — and maximize $P(\text{evidence} \mid \text{scenario})\,P(\text{scenario}) = P(\text{evidence} \cap \text{scenario})$: the **joint probability**, which is what we truly care about. It is based on the seventh video of Lesson 6 (Point Estimation), the doorway to maximum a posteriori estimation.

## 1. Introduction

MLE crowns whichever scenario best *explains* the evidence. But explanations aren't free: some scenarios are common, some are absurd, and an absurd scenario that explains the evidence perfectly should not automatically beat a common one that explains it well. This notebook makes that instinct mathematical — and recognizes the resulting formula as an old friend from Lesson 1.

After completing this notebook you will be able to:

- Construct the counterexample where plain MLE picks an absurd winner.
- Repair it by weighting likelihoods with **priors**: maximize $P(\text{evidence} \mid \text{scenario})\,P(\text{scenario})$.
- Recognize the product as the **joint probability** $P(\text{evidence} \cap \text{scenario})$ via the conditional-probability rule.
- Explore when priors flip verdicts and when evidence overwhelms them.

## 2. Theory

### The new suspect

The original contest: movies (high $P(\text{popcorn} \mid \cdot)$), board games (medium), nap (low) — MLE picked movies. Now two candidates:

- **Movies:** creates popcorn on the floor with **high** probability.
- **Popcorn-throwing contest:** creates popcorn on the floor with **very, very high** probability — almost for sure.

Pure likelihood crowns the contest. However, something tells us movies should win: *watching a movie is likely; a popcorn-throwing contest is a very unlikely event.* Even though the contest is more likely to have **generated the evidence**, it is far less likely to have **happened at all** — and that should matter.

### The repair: multiply by the prior

Before, we maximized the conditional probability $P(\text{popcorn} \mid \text{scenario})$, where

$$
P(\text{popcorn} \mid \text{movies}) < P(\text{popcorn} \mid \text{contest})
$$

But the scenarios' own probabilities point the other way, much more strongly:

$$
P(\text{movies}) \gg P(\text{contest})
$$

Factor that in by **multiplying**:

$$
P(\text{popcorn} \mid \text{movies})\,P(\text{movies}) \;>\; P(\text{popcorn} \mid \text{contest})\,P(\text{contest})
$$

The inequality flips — movies wins again, now for the right reason.

### This product is a joint probability

The formula looks familiar from Lesson 1: $P(A \mid B)\,P(B) = P(A \cap B)$. So what we are maximizing is

$$
P(\text{popcorn} \cap \text{movies}) \quad\text{vs}\quad P(\text{popcorn} \cap \text{contest})
$$

— *the probability that the evidence and the scenario happen together*. And that makes more sense: not just "which scenario explains the popcorn best," but "which popcorn-producing world is most probable overall." (Divide by the constant $P(\text{popcorn})$ and this is precisely the posterior $P(\text{scenario} \mid \text{popcorn})$ — Bayes' theorem; the next video names the resulting estimator **MAP**.)

### The moral

MLE is the special case where all scenarios get equal priors. When priors are genuinely unequal — rare diseases, absurd contests, extreme parameter values — the joint/posterior criterion corrects MLE's blind spot. And the correction fades as evidence accumulates: enough popcorn *everywhere, every day* would eventually make even a contest plausible.

## 3. Worked Examples

### Example 1: with numbers

Say $P(\text{popcorn}\mid\text{movies}) = 0.7$, $P(\text{popcorn}\mid\text{contest}) = 0.99$, and priors $P(\text{movies}) = 0.3$, $P(\text{contest}) = 0.001$. Then:

$$
0.7 \times 0.3 = 0.21 \qquad\text{vs}\qquad 0.99 \times 0.001 = 0.00099
$$

Movies wins by a factor of ~212, despite losing the pure-likelihood contest 0.7 to 0.99.

### Example 2: the flip threshold

The contest would need a prior above $\frac{0.7 \times 0.3}{0.99} \approx 0.212$ — i.e., popcorn-throwing contests happening about once in five evenings — before the joint criterion would crown it. Priors don't forbid unlikely scenarios; they demand proportionate evidence.

### Example 3: the disease-test reunion

This is exactly Lesson 1's rare-disease logic: a test (evidence) that's 99% sensitive still usually indicates a healthy patient if the disease has prior 0.1%. MLE-thinking ("the disease best explains a positive test") is the base-rate fallacy; multiplying by the prior is the cure — in both diagnosis and estimation.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

scenarios = {
    #                likelihood P(popcorn|s)   prior P(s)
    "movies":        (0.70,                    0.300),
    "board games":   (0.30,                    0.200),
    "nap":           (0.05,                    0.400),
    "popcorn contest": (0.99,                  0.001),
}

### MLE vs the joint criterion

In [ ]:
print(f"{'scenario':<16} {'likelihood':>10} {'prior':>7} {'joint':>9}")
for s, (lik, prior) in scenarios.items():
    print(f"{s:<16} {lik:>10} {prior:>7} {lik * prior:>9.5f}")

mle = max(scenarios, key=lambda s: scenarios[s][0])
joint = max(scenarios, key=lambda s: scenarios[s][0] * scenarios[s][1])
print(f"\nMLE winner (max likelihood): {mle}")
print(f"joint winner (max likelihood x prior): {joint}")

### The joint really is P(evidence AND scenario): simulate the world

Sample evenings according to the priors, let each scenario produce (or not) popcorn by its likelihood, and count which (popcorn, scenario) pairs occur — the frequencies match the products.

In [ ]:
rng = np.random.default_rng(0)

names = list(scenarios)
priors = np.array([scenarios[s][1] for s in names])
liks = np.array([scenarios[s][0] for s in names])
priors_norm = priors / priors.sum()          # make the evening scenarios exhaustive

n = 1_000_000
which = rng.choice(len(names), p=priors_norm, size=n)
popcorn = rng.random(n) < liks[which]

print(f"{'scenario':<16} {'P(popcorn AND s) sim':>21} {'lik x prior':>12}")
for i, s in enumerate(names):
    sim = np.mean(popcorn & (which == i))
    print(f"{s:<16} {sim:>21.5f} {liks[i] * priors_norm[i]:>12.5f}")

among = which[popcorn]
best = names[np.bincount(among).argmax()]
print(f"\nAmong evenings that DID end with popcorn, the most common cause: {best}")

### When does the contest win? The prior threshold

In [ ]:
contest_priors = np.logspace(-4, -0.3, 300)
movies_joint = 0.70 * 0.300
contest_joint = 0.99 * contest_priors
threshold = movies_joint / 0.99

plt.figure(figsize=(9, 4.5))
plt.axhline(movies_joint, color="tab:green", linewidth=2, label="movies: 0.7 x 0.3")
plt.plot(contest_priors, contest_joint, color="tab:red", linewidth=2,
         label="contest: 0.99 x prior")
plt.axvline(threshold, color="gray", linestyle=":",
            label=f"flip at prior = {threshold:.3f}")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("P(popcorn contest)  (log)")
plt.ylabel("joint probability  (log)")
plt.title("The absurd explanation needs a ~21% prior before it can win")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 5. Visualizations

### Likelihood, prior, joint — the flip made visible

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.2))
pair = ["movies", "popcorn contest"]
colors = ["tab:green", "tab:red"]

for ax, title, values in [
        (axes[0], "likelihood P(popcorn | s)", [scenarios[s][0] for s in pair]),
        (axes[1], "prior P(s)", [scenarios[s][1] for s in pair]),
        (axes[2], "joint = likelihood x prior", [scenarios[s][0] * scenarios[s][1] for s in pair])]:
    ax.bar(pair, values, color=colors, edgecolor="black", alpha=0.85)
    ax.set_title(title)
    ax.grid(True, alpha=0.3, axis="y")
    for i, v in enumerate(values):
        ax.annotate(f"{v:.5g}", xy=(i, v), ha="center", va="bottom")
axes[1].set_yscale("log")
axes[2].set_yscale("log")
plt.suptitle("The contest wins round 1, loses round 2 by far more -- the joint flips the verdict")
plt.tight_layout()
plt.show()

### Evidence vs prior: who wins as popcorn keeps appearing

If popcorn shows up on $k$ *independent* evenings-worth of evidence, likelihoods compound ($0.7^k$ vs $0.99^k$) while priors stay fixed. Eventually even a tiny prior is overcome — priors matter most when evidence is scarce.

In [ ]:
ks = np.arange(1, 61)
movies_k = 0.300 * 0.70**ks
contest_k = 0.001 * 0.99**ks

plt.figure(figsize=(9.5, 4.5))
plt.plot(ks, movies_k, color="tab:green", linewidth=2, label="movies joint")
plt.plot(ks, contest_k, color="tab:red", linewidth=2, label="contest joint")
crossover = ks[np.argmax(contest_k > movies_k)]
plt.axvline(crossover, color="gray", linestyle=":",
            label=f"crossover at k = {crossover} popcorn observations")
plt.yscale("log")
plt.xlabel("number of independent popcorn observations k")
plt.ylabel("joint probability (log)")
plt.title("Accumulating evidence eventually overwhelms the prior")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 6. Exercises

### Basic

**Exercise 1.** Muddy paw prints again: $P(\text{prints}\mid\text{dog}) = 0.8$ with prior 0.5, versus a new suspect — an escaped zoo bear, $P(\text{prints}\mid\text{bear}) = 0.999$ with prior $10^{-5}$. Compute both joints and give the verdict.

**Exercise 2.** State the identity used in the video ($P(A\mid B)P(B) = P(A\cap B)$) and explain in one sentence why the joint — not the bare conditional — is "what we actually want to maximize."

### Intermediate

**Exercise 3.** With the four-scenario table above, compute the full **posterior** $P(s \mid \text{popcorn})$ for each scenario (normalize the joints). Check the posteriors sum to 1 and compare the ranking with the joints' — why is it necessarily the same?

**Exercise 4.** Solve the crossover of the compounding-evidence plot analytically: for priors $\pi_m, \pi_c$ and likelihoods $0.7, 0.99$ per observation, find the $k$ where $\pi_c\,0.99^k = \pi_m\,0.7^k$, and verify against the plot.

### Advanced

**Exercise 5.** Priors as regularization, first taste: for the coin from the Bernoulli notebook with data 5-of-5 heads, compare the pure MLE ($\hat p = 1$) with the joint-maximizer when the prior over $p$ is the fair-ish density $\propto p(1-p)$. Grid-maximize $p^5 \cdot p(1-p)$ and interpret the pullback.

**Exercise 6.** When priors mislead: keep the popcorn numbers but suppose your prior for "contest" was wrong — it's actually a college dorm where contests happen weekly ($\pi_c = 0.15$). Recompute, and discuss: which is riskier in practice, a wrong prior with strong evidence, or a wrong prior with weak evidence? Support your answer with the compounding-evidence formula.

## 7. Solutions

In [ ]:
# Exercise 1: dog vs bear
dog = 0.8 * 0.5
bear = 0.999 * 1e-5
print(f"Exercise 1: dog joint = {dog}, bear joint = {bear:.2e}")
print(f"The dog wins by {dog / bear:,.0f}x -- near-certain evidence generation")
print("cannot rescue a one-in-a-hundred-thousand suspect.")

In [ ]:
# Exercise 2: the identity
print("Exercise 2: P(A|B) P(B) = P(A and B), the conditional-probability rule.")
print("We want the scenario most probable IN CONJUNCTION with the observed")
print("evidence -- the most probable popcorn-containing world -- not merely the")
print("scenario that would explain popcorn best if it had happened.")

In [ ]:
# Exercise 3: full posterior
joints = {s: scenarios[s][0] * scenarios[s][1] for s in scenarios}
total = sum(joints.values())
print(f"{'scenario':<16} {'joint':>9} {'posterior':>10}")
for s, j in joints.items():
    print(f"{s:<16} {j:>9.5f} {j / total:>10.4f}")
print(f"posteriors sum to {sum(j / total for j in joints.values()):.4f}")
print("Normalizing divides every joint by the same constant P(popcorn), so the")
print("ranking -- and the argmax -- cannot change: maximizing the joint IS")
print("maximizing the posterior.")

In [ ]:
# Exercise 4: the crossover, analytically
# pi_c * 0.99^k = pi_m * 0.7^k  =>  k = ln(pi_m/pi_c) / ln(0.99/0.7)
k_star = np.log(0.300 / 0.001) / np.log(0.99 / 0.7)
print(f"Exercise 4: k* = ln(300) / ln(0.99/0.7) = {k_star:.1f}")
print(f"so the contest overtakes at k = {int(np.ceil(k_star))} -- matching the plot.")

In [ ]:
# Exercise 5: a prior tames the 5-of-5 coin
p = np.linspace(0.001, 0.999, 999)
mle_obj = p**5
map_obj = p**5 * p * (1 - p)
print(f"Exercise 5: pure MLE argmax:      p = {p[np.argmax(mle_obj)]:.3f}")
print(f"with prior p(1-p): argmax at     p = {p[np.argmax(map_obj)]:.3f}")
print("The prior's (1-p) factor vetoes p = 1 (a certain-heads coin was deemed")
print("a priori impossible-ish) and pulls the estimate to 6/7 ~ 0.857 -- the")
print("regularization effect the coming videos formalize as MAP.")

In [ ]:
# Exercise 6: a wrong prior, weak vs strong evidence
pi_c_dorm = 0.15
print(f"Exercise 6: dorm recomputation: movies {0.7 * 0.3:.3f} vs "
      f"contest {0.99 * pi_c_dorm:.3f} -> contest wins already at k = 1.")
print()
print("Risk analysis: with WEAK evidence (small k) the prior dominates the")
print("joint pi * L^k, so a wrong prior decides the verdict -- dangerous. With")
print("STRONG evidence the L^k factor dwarfs any fixed prior (the crossover")
print("formula k* = ln(pi ratio)/ln(L ratio) is finite), so wrong priors get")
print("corrected. Priors are most helpful -- and most hazardous -- at small n.")

## 8. Conclusion

- Pure MLE has a blind spot: the popcorn-throwing contest explains the evidence better than movies ($0.99 > 0.7$) yet is absurd *a priori* — likelihood alone can crown absurdity.
- The repair: weight each scenario's likelihood by its own probability and maximize $P(\text{evidence} \mid s)\,P(s)$ — which, by the conditional-probability rule, is the **joint probability** $P(\text{evidence} \cap s)$: the most probable *world*, not the most flattering explanation.
- The numbers flip decisively ($0.21$ vs $0.00099$), and normalizing by $P(\text{evidence})$ shows this is exactly maximizing the Bayesian **posterior** — Lesson 1's disease-test logic, now as an estimation principle.
- Priors matter most when evidence is scarce and fade as it compounds ($\pi L^k$: any finite prior gap is overcome by enough observations) — and a prior over parameters (e.g. $p(1-p)$ on a coin) tames MLE's overfit edge cases.

Next: this criterion gets its name — **maximum a posteriori (MAP) estimation** — and its ML identity: regularization.